In [42]:
from src import *
import numpy as np
from tqdm import tqdm
from time import sleep
import os

from myutils import email_notify


from datetime import datetime
today = datetime.strftime(datetime.today(), '%Y%m%d')
del datetime

In [43]:
today

'20241103'

In [44]:
# with RaspiLED() as led:
#     led.check()

In [45]:
np.linspace(0.1, 0.4, 61)[31:41]

array([0.255, 0.26 , 0.265, 0.27 , 0.275, 0.28 , 0.285, 0.29 , 0.295,
       0.3  ])

In [46]:
sampling_rate = 20 #Hz
freq_list = np.linspace(0.1, 0.4, 61)[31:41]

pic_time = (1 / (sampling_rate * freq_list) / 2 * 1e6).astype(int)

A = 5
samples = 50
repeat = 200

interval = 50e-3
exposure_time = 500e-6 #500 us

measurement = 'SPADE'

In [47]:
############
# NO NOISE #
############

@email_notify('hcnzj@qq.com')
def main():
    with EasyDcam() as dcam, EasyALP4() as alp:
        for i, picture_time in enumerate(pic_time):
            print(f'({i + 1}): Current sensor temperature is {dcam.ez_temperature()}')
            if dcam.ez_temperature() >= -30:
                raise RuntimeError("qCMOS's temperature is too high.")

            ground_truth = np.round(freq_list[i], 5)

            alp.ez_load_seq([alp.ez_single_pixel(0), alp.ez_single_pixel(A)], picture_time)
            dcam.ez_exposure_time(exposure_time)
            dcam.ez_triggersource_masterpluse(samples, interval)

            if measurement.upper() == 'SPADE':
                dcam.ez_roi(**SPADE.ROI)
            elif measurement.upper() == 'DI':
                dcam.ez_roi(**DI.ROI)

            raw, timestamp = [], []
            for _ in tqdm(range(repeat)):
                dcam.buf_alloc(samples)
                dcam.cap_snapshot()

                alp.Run()
                sleep(1e-6)
                dcam.cap_firetrigger()

                dcam.ez_wait_capture()

                dcam.cap_stop()
                alp.Halt()

                raw_, timestamp_ = [], []
                for frame in range(samples):
                    framedata_ = dcam.ez_read_buf(frame)
                    raw_.append(framedata_[0])
                    timestamp_.append(framedata_[1])

                dcam.buf_release()

                raw.append(raw_)
                timestamp.append(timestamp_)

            raw = np.array(raw)
            timestamp = np.array(timestamp)

            if not os.path.exists(f'__raw__/{today}'):
                os.makedirs(f'__raw__/{today}')
            if not os.path.exists(f'__estimates__/{today}'):
                os.makedirs(f'__estimates__/{today}')

            metadata = MetaData(measurement, ground_truth, A*DMD.PIXEL_SIZE/2, timestamp)
            est = FrequencyEstimation(np.array(raw), metadata)
            est.savez(f'./__estimates__/{today}/{measurement.lower()}_{ground_truth}.npz')

            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_raw.npy', raw)
            np.save(f'./__raw__/{today}/{measurement.lower()}_{ground_truth}_timestamp.npy', timestamp)


if __name__ == '__main__':
    main()

qCMOS found, current sensor temperature is -37.0.
DMD found, resolution = 1024 x 768.
(1): Current sensor temperature is -37.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(2): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(3): Current sensor temperature is -36.0


...

(9): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


(10): Current sensor temperature is -36.0


100%|██████████| 200/200 [09:05<00:00,  2.73s/it]


EasyALP4 exited
EasyDcam exited


In [48]:
raise RuntimeError('STOP HERE')

RuntimeError: STOP HERE